In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import jit
from ipywidgets import interact,IntSlider

# First we write the functions for our CFD solver

In [ ]:
# Linear interpolate function
@jit(nopython=True,fastmath=True)
def linear_interp(val1,dx1,val2,dx2):
    return (val1*dx2+val2*dx1)/(dx1+dx2)

# CFD solver
@jit(nopython=True)
def CFD_EDACM(P,U,V,rho,nu,beta,dx,dy):
    m=len(dx)
    n=len(dy)
    der_p=np.zeros_like(P)
    der_u=np.zeros_like(U)
    der_v=np.zeros_like(V)
    for i in range(1,m-1):
        for j in range(1,n-1):
            # First let us interpolate the values at the faces
            P_east=linear_interp(P[i,j],dx[i],P[i+1,j],dx[i+1])
            P_west=linear_interp(P[i-1,j],dx[i-1],P[i,j],dx[i])
            P_north=linear_interp(P[i,j],dy[j],P[i,j+1],dy[j+1])
            P_south=linear_interp(P[i,j-1],dy[j-1],P[i,j],dy[j])
            
            U_east=linear_interp(U[i,j],dx[i],U[i+1,j],dx[i+1])
            U_west=linear_interp(U[i-1,j],dx[i-1],U[i,j],dx[i])
            U_north=linear_interp(U[i,j],dy[j],U[i,j+1],dy[j+1])
            U_south=linear_interp(U[i,j-1],dy[j-1],U[i,j],dy[j])
            
            V_east=linear_interp(V[i,j],dx[i],V[i+1,j],dx[i+1])
            V_west=linear_interp(V[i-1,j],dx[i-1],V[i,j],dx[i])
            V_north=linear_interp(V[i,j],dy[j],V[i,j+1],dy[j+1])
            V_south=linear_interp(V[i,j-1],dy[j-1],V[i,j],dy[j])
            
            # Writing the convective terms
            con_p_x=beta*(U_east-U_west)/dx[i]
            con_p_y=beta*(V_north-V_south)/dy[j]
            con_u_x=(U_east**2-U_west**2)/dx[i]
            con_u_y=(U_north*V_north-U_south*V_south)/dy[j]
            con_v_x=(U_east*V_east-U_west*V_west)/dx[i]
            con_v_y=(V_north**2-V_south**2)/dy[j]
            pressure_u=(P_east-P_west)/(rho*dx[i])
            pressure_v=(P_north-P_south)/(rho*dy[j])
            
            # Now we find the gradients at the faces and write the diffusion terms 
            dpdx_e=(P[i+1,j]-P[i,j])/(0.5*(dx[i]+dx[i+1]))
            dpdx_w=(P[i,j]-P[i-1,j])/(0.5*(dx[i-1]+dx[i]))
            dpdy_n=(P[i,j+1]-P[i,j])/(0.5*(dy[j]+dy[j+1]))
            dpdy_s=(P[i,j]-P[i,j-1])/(0.5*(dy[j-1]+dy[j]))
            diff_p_x=nu*(dpdx_e-dpdx_w)/dx[i]
            diff_p_y=nu*(dpdy_n-dpdy_s)/dy[j]
            
            dudx_e=(U[i+1,j]-U[i,j])/(0.5*(dx[i]+dx[i+1]))
            dudx_w=(U[i,j]-U[i-1,j])/(0.5*(dx[i-1]+dx[i]))
            dudy_n=(U[i,j+1]-U[i,j])/(0.5*(dy[j]+dy[j+1]))
            dudy_s=(U[i,j]-U[i,j-1])/(0.5*(dy[j-1]+dy[j]))
            diff_u_x=nu*(dudx_e-dudx_w)/dx[i]
            diff_u_y=nu*(dudy_n-dudy_s)/dy[j]
            
            dvdx_e=(V[i+1,j]-V[i,j])/(0.5*(dx[i]+dx[i+1]))
            dvdx_w=(V[i,j]-V[i-1,j])/(0.5*(dx[i-1]+dx[i]))
            dvdy_n=(V[i,j+1]-V[i,j])/(0.5*(dy[j]+dy[j+1]))
            dvdy_s=(V[i,j]-V[i,j-1])/(0.5*(dy[j-1]+dy[j]))
            diff_v_x=nu*(dvdx_e-dvdx_w)/dx[i]
            diff_v_y=nu*(dvdy_n-dvdy_s)/dy[j]
            
            
            der_p[i,j]=diff_p_x+diff_p_y-con_p_x-con_p_y
            der_u[i,j]=diff_u_x+diff_u_y-con_u_x-con_u_y-pressure_u
            der_v[i,j]=diff_v_x+diff_v_y-con_v_x-con_v_y-pressure_v
            
    return der_p,der_u,der_v
          
            

# Now we write immersed boundary methods

In [ ]:
@jit(nopython=True)
def IBM_forcing_der(U,V,x_c,y_c,lag_x,lag_y,timestep):
    der_u=np.zeros_like(U)
    der_v=np.zeros_like(V)
    m=len(x_c)
    n=len(y_c)
    neigh=3
    # iterating over all the lagrangian points
    for lag_ind in range(0,len(lag_x),1):
        lx=lag_x[lag_ind]
        ly=lag_y[lag_ind]
        # finding the nearest neighbours
        near_i=0
        near_j=0
        for i in range(0,m-1):
            if x_c[i]<lx and lx<=x_c[i+1]:
                near_i=i+0
                break
        for j in range(0,n-1):
            if y_c[j]<ly and ly<=y_c[j+1]:
                near_j=j+0
                break
        #Make calculations for the nearest neighbours
        U_int=0.0
        V_int=0.0
        dx_h=x_c[near_i+1]-x_c[near_i]
        dy_h=y_c[near_j+1]-y_c[near_j]
        #Interpolation Operation
        for i_ind in range(near_i-neigh+1,near_i+neigh+1):
            for j_ind in range(near_j-neigh+1,near_j+neigh+1):
                x_eu=x_c[i_ind]
                y_eu=y_c[j_ind]
                rx=abs(x_eu-lx)/dx_h
                ry=abs(y_eu-ly)/dy_h
                delta_x=0.0
                delta_y=0.0
                if rx<=2.0:
                    delta_x=0.25*(1+np.cos(0.5*np.pi*rx))
                if ry<=2.0:
                    delta_y=0.25*(1+np.cos(0.5*np.pi*ry))
                delta=delta_x*delta_y
                U_int+=delta*U[i_ind,j_ind]
                V_int+=delta*V[i_ind,j_ind]
            
        # Forcing computation
        fx=(-1.0)*U_int/timestep
        fy=(-1.0)*V_int/timestep
        
        # Spreading Operation
        for i_ind in range(near_i-neigh+1,near_i+neigh+1):
            for j_ind in range(near_j-neigh+1,near_j+neigh+1):
                x_eu=x_c[i_ind]
                y_eu=y_c[j_ind]
                rx=abs(x_eu-lx)/dx_h
                ry=abs(y_eu-ly)/dy_h
                delta_x=0.0
                delta_y=0.0
                if rx<=2.0:
                    delta_x=0.25*(1+np.cos(0.5*np.pi*rx))
                if ry<=2.0:
                    delta_y=0.25*(1+np.cos(0.5*np.pi*ry))
                delta=delta_x*delta_y
                der_u[i_ind,j_ind]+=delta*fx
                der_v[i_ind,j_ind]+=delta*fy      
                
    return der_u,der_v

# We write the force computations

In [ ]:
# Compute net force on body
@jit(nopython=True)
def IBM_force_computation(U,V,x_c,y_c,lag_x,lag_y,timestep,rho):
    m=len(x_c)
    n=len(y_c)
    neigh=3
    f_x=0.0
    f_y=0.0
    # iterating over all the lagrangian points
    for lag_ind in range(0,len(lag_x),1):
        lx=lag_x[lag_ind]
        ly=lag_y[lag_ind]
        # finding the nearest neighbours
        near_i=0
        near_j=0
        for i in range(0,m-1):
            if x_c[i]<lx and lx<=x_c[i+1]:
                near_i=i+0
                break
        for j in range(0,n-1):
            if y_c[j]<ly and ly<=y_c[j+1]:
                near_j=j+0
                break
        #Make calculations for the nearest neighbours
        U_int=0.0
        V_int=0.0
        dx_h=x_c[near_i+1]-x_c[near_i]
        dy_h=y_c[near_j+1]-y_c[near_j]
        #Interpolation Operation
        for i_ind in range(near_i-neigh+1,near_i+neigh+1):
            for j_ind in range(near_j-neigh+1,near_j+neigh+1):
                x_eu=x_c[i_ind]
                y_eu=y_c[j_ind]
                rx=abs(x_eu-lx)/dx_h
                ry=abs(y_eu-ly)/dy_h
                delta_x=0.0
                delta_y=0.0
                if rx<=2.0:
                    delta_x=0.25*(1+np.cos(0.5*np.pi*rx))
                if ry<=2.0:
                    delta_y=0.25*(1+np.cos(0.5*np.pi*ry))
                delta=delta_x*delta_y
                U_int+=delta*U[i_ind,j_ind]
                V_int+=delta*V[i_ind,j_ind]
            
        # Forcing computation
        f_x+=(rho*dx_h*dy_h)*U_int/timestep
        f_y+=(rho*dx_h*dy_h)*V_int/timestep
        
    return f_x,f_y

# Now writing the time integrators

In [ ]:
# TVD Runge Kutta function
def TVD_RK2_integrate(BC_function,P_state,U_state,V_state,U_bound,P_bound,timestep,rho,nu,beta,dx,dy):
    # Implement second order TVD RK for solver only
    dp,du,dv=CFD_EDACM(P_state,U_state,V_state,rho,nu,beta,dx,dy)
    P_temp=P_state+timestep*dp
    U_temp=U_state+timestep*du
    V_temp=V_state+timestep*dv
    # Callling the boundary condition function
    # For our case U_bound is the velocity for wall
    # P is the fixed pressure at a point
    P_temp,U_temp,V_temp=BC_function(P_temp,U_temp,V_temp,U_bound,P_bound)
    dp,du,dv=CFD_EDACM(P_temp,U_temp,V_temp,rho,nu,beta,dx,dy)
    P_new=0.5*(P_state+P_temp)+0.5*timestep*dp
    U_new=0.5*(U_state+U_temp)+0.5*timestep*du
    V_new=0.5*(V_state+V_temp)+0.5*timestep*dv
    P_new,U_new,V_new=BC_function(P_new,U_new,V_new,U_bound,P_bound)
    
    return P_new,U_new,V_new

In [ ]:
# TVD RK2 with IBM Forcing
def IBM_RK2(BC_function,P_state,U_state,V_state,U_bound,P_bound,timestep,rho,nu,beta,dx,dy,x_c,y_c,lag_x,lag_y):
    
    dp,du1,dv1=CFD_EDACM(P_state,U_state,V_state,rho,nu,beta,dx,dy)
    du2,dv2=IBM_forcing_der(U_state,V_state,x_c,y_c,lag_x,lag_y,timestep)
    
    P_temp=P_state+timestep*dp
    U_temp=U_state+timestep*(du1+du2)
    V_temp=V_state+timestep*(dv1+dv2)
    
    P_temp,U_temp,V_temp=BC_function(P_temp,U_temp,V_temp,U_bound,P_bound)
    
    dp,du1,dv1=CFD_EDACM(P_temp,U_temp,V_temp,rho,nu,beta,dx,dy)
    du2,dv2=IBM_forcing_der(U_temp,V_temp,x_c,y_c,lag_x,lag_y,timestep)
    
    P_new=0.5*(P_state+P_temp)+0.5*timestep*dp
    U_new=0.5*(U_state+U_temp)+0.5*timestep*(du1+du2)
    V_new=0.5*(V_state+V_temp)+0.5*timestep*(dv1+dv2)
    
    P_new,U_new,V_new=BC_function(P_new,U_new,V_new,U_bound,P_bound)
    
    return P_new,U_new,V_new
    

# Now writing the boundary conditions function

In [ ]:
def Lid_bc(P,U,V,U_wall,P_fix):
    m,n=np.shape(P)
    # Wall 1 bc
    P[0,:]=P[1,:]
    U[0,:]=(-1.0)*U[1,:]
    V[0,:]=(-1.0)*V[1,:]
    
    # Wall 2 bc 
    P[m-1,:]=P[m-2,:]
    U[m-1,:]=(-1.0)*U[m-2,:]
    V[m-1,:]=(-1.0)*V[m-2,:]
    
    # Wall 3 bc
    P[:,0]=P[:,1]
    U[:,0]=(-1.0)*U[:,1]
    V[:,0]=(-1.0)*V[:,1]
    
    # Wall 4 bc
    P[:,n-1]=P[:,n-2]
    U[:,n-1]=(2.0*U_wall)-U[:,n-2]
    V[:,n-1]=(-1.0)*V[:,n-2]
    
    # Fixing pressure at the bottom node to be 1
    P[2,2]=P_fix
    
    return P,U,V

# Computing divergence function

In [ ]:
@jit(nopython=True,fastmath=True)
def compute_divergence(U,V,dx,dy):
    div_mat=np.zeros_like(U)
    m=len(dx)
    n=len(dy)
    for i in range(2,m-2):
        for j in range(2,n-2):
            # First let us interpolate the values at the faces
            U_east=linear_interp(U[i,j],dx[i],U[i+1,j],dx[i+1])
            U_west=linear_interp(U[i-1,j],dx[i-1],U[i,j],dx[i])
            V_north=linear_interp(V[i,j],dy[j],V[i,j+1],dy[j+1])
            V_south=linear_interp(V[i,j-1],dy[j-1],V[i,j],dy[j])
            
            div_mat[i,j]=((U_east-U_west)/dx[i])+((V_north-V_south)/dy[j]);
    return div_mat
    

# Now we create functions that help in generating the Euler domain

In [ ]:
def init_domains(x_eu,y_eu,U_in,P_out):
    x_dom=np.concatenate([[2.0*x_eu[0]-x_eu[1]],x_eu,[2.0*x_eu[-1]-x_eu[-2]]])
    y_dom=np.concatenate([[2.0*y_eu[0]-y_eu[1]],y_eu,[2.0*y_eu[-1]-y_eu[-2]]])
    
    x_c=0.5*(x_dom[1:]+x_dom[:-1])
    y_c=0.5*(y_dom[1:]+y_dom[:-1])
    dx=x_dom[1:]-x_dom[:-1]
    dy=y_dom[1:]-y_dom[:-1]
    m=len(dx)
    n=len(dy)
    xx,yy=np.meshgrid(x_c,y_c,indexing="ij")
    U=U_in*np.ones_like(xx)
    V=np.zeros_like(xx)
    P=P_out*np.ones_like(xx)
    
    return x_c,y_c,dx,dy,xx,yy,P,U,V
    

# Now let us test the code

In [ ]:
length=1.0
x=np.concatenate([np.linspace(0,length/2.0,120),np.linspace(length/2.0,length,60)[1:]])
U_in=1.0
P_out=0.0
x_c,y_c,dx,dy,xx,yy,P,U,V=init_domains(x,x,U_in,P_out)

In [ ]:
rho=1.0
Re=400
nu=U_in*length/Re
beta=500

In [ ]:
n_iter=700000
n_print=1000
n_save=7000
timestep=1e-5

In [ ]:
P_array=[]
U_array=[]
V_array=[]
for i in range(n_iter):
    P,U,V=TVD_RK2_integrate(Lid_bc,P,U,V,U_in,P_out,timestep,rho,nu,beta,dx,dy)
    if i%n_print==0:
        print("Iteration "+str(i+1)+"completed")
        div_mat=compute_divergence(U,V,dx,dy)
        print("The maximum value of divergence is :"+str(np.max(np.abs(div_mat))))
    if i%n_save==0:
        print("Iteration "+str(i+1)+"saved")
        P_array.append(np.copy(P))
        U_array.append(np.copy(U))
        V_array.append(np.copy(V))


In [ ]:
def plotter(ind):
    plt.contourf(xx,yy,U_array[ind],levels=50)
    #plt.streamplot(xx.T,yy.T,U_array[ind].T,V_array[ind].T)
    plt.colorbar()
    
D=interact(plotter,ind=IntSlider(min=0,max=len(U_array)-1,step=1))

In [ ]:
y_arr=yy.T[:,50]
u_arr=U_array[99].T[:,130]
plt.plot(y_arr,u_arr)

In [ ]:
ghia = np.array(np.mat('1.0000  1.00000  1.00000  1.00000  1.00000  1.00000  1.00000  1.00000;0.9766  0.84123  0.75837  0.65928  0.53236  0.48223  0.47244  0.47221;0.9688  0.78871  0.68439  0.57492  0.48296  0.46120  0.47048  0.47783;0.9609  0.73722  0.61756  0.51117  0.46547  0.45992  0.47323  0.48070;0.9531  0.68717  0.55892  0.46604  0.46101  0.46036  0.47167  0.47804;0.8516  0.23151  0.29093  0.33304  0.34682  0.33556  0.34228  0.34635;0.7344  0.00332  0.16256  0.18719  0.19791  0.20087  0.20591  0.20673;0.6172 -0.13641  0.02135  0.05702  0.07156  0.08183  0.08342  0.08344;0.5000 -0.20581 -0.11477 -0.06080 -0.04272 -0.03039 -0.03800  0.03111;0.4531 -0.21090 -0.17119 -0.10648 -0.86636 -0.07404 -0.07503 -0.07540;0.2813 -0.15662 -0.32726 -0.27805 -0.24427 -0.22855 -0.23176 -0.23186;0.1719 -0.10150 -0.24299 -0.38289 -0.34323 -0.33050 -0.32393 -0.32709;0.1016 -0.06434 -0.14612 -0.29730 -0.41933 -0.40435 -0.38324 -0.38000;0.0703 -0.04775 -0.10338 -0.22220 -0.37827 -0.43643 -0.43025 -0.41657;0.0625 -0.04192 -0.09266 -0.20196 -0.35344 -0.42901 -0.43590 -0.42537;0.0547 -0.03717 -0.08186 -0.18109 -0.32407 -0.41165 -0.43154 -0.42735;0.0000  0.00000  0.00000  0.00000  0.00000  0.00000  0.00000  0.00000'))
plt.figure()
plt.plot(ghia[:,2], ghia[:,0],'o')
plt.plot(u_arr,y_arr,'-')
plt.show()

# Now we use the code for Channel Flow

In [ ]:
def Channel_bc(P,U,V,U_in,P_fix):
    m,n=np.shape(P)
    # Inlet bc
    P[0,:]=P[1,:]
    U[0,:]=2.0*U_in-U[1,:]
    V[0,:]=(-1.0)*V[1,:]
    
    # Outlet bc 
    P[m-1,:]=P_fix
    U[m-1,:]=(1.0)*U[m-2,:]
    V[m-1,:]=(1.0)*V[m-2,:]
    
    # Wall 3 bc
    P[:,0]=P[:,1]
    U[:,0]=(-1.0)*U[:,1]
    V[:,0]=(-1.0)*V[:,1]
    
    # Wall 4 bc
    P[:,n-1]=P[:,n-2]
    U[:,n-1]=(-1.0)*U[:,n-2]
    V[:,n-1]=(-1.0)*V[:,n-2]
    

    
    return P,U,V

In [ ]:
diameter=2
length=20*diameter
x=np.linspace(0,length,200)
y=np.linspace(0,diameter,20)
U_in=1.0
P_out=0.0
x_c,y_c,dx,dy,xx,yy,P,U,V=init_domains(x,y,U_in,P_out)
print(len(dx),len(dy))
rho=1.0
Re=400
nu=U_in*diameter/Re
beta=500
n_iter=700000
n_print=1000
n_save=7000
timestep=1e-5

P_array=[]
U_array=[]
V_array=[]
for i in range(n_iter):
    P,U,V=TVD_RK2_integrate(Channel_bc,P,U,V,U_in,P_out,timestep,rho,nu,beta,dx,dy)
    if i%n_print==0:
        print("Iteration "+str(i+1)+"completed")
        div_mat=compute_divergence(U,V,dx,dy)
        print("The maximum value of divergence is :"+str(np.max(np.abs(div_mat))))
    if i%n_save==0:
        print("Iteration "+str(i+1)+"saved")
        P_array.append(np.copy(P))
        U_array.append(np.copy(U))
        V_array.append(np.copy(V))


In [ ]:
def plotter(ind):
    plt.contourf(xx,yy,U_array[ind])
    #plt.streamplot(xx.T,yy.T,U_array[ind].T,V_array[ind].T)
    plt.colorbar()
    
D=interact(plotter,ind=IntSlider(min=0,max=len(U_array)-1,step=1))

In [ ]:
y_arr=yy.T[:,50]
u_arr=U_array[99].T[:,200]
plt.plot(y_arr,u_arr)

# Solving for external flow problems

In [ ]:
def External_flow_bc(P,U,V,U_in,P_fix):
    m,n=np.shape(P)
    # Inlet bc
    P[0,:]=P[1,:]
    U[0,:]=2.0*U_in-U[1,:]
    V[0,:]=(-1.0)*V[1,:]
    
    # Outlet bc 
    P[m-1,:]=P_fix
    U[m-1,:]=(1.0)*U[m-2,:]
    V[m-1,:]=(1.0)*V[m-2,:]
    
    # Wall 3 bc
    P[:,0]=P[:,1]
    U[:,0]=(1.0)*U[:,1]
    V[:,0]=(-1.0)*V[:,1]
    
    # Wall 4 bc
    P[:,n-1]=P[:,n-2]
    U[:,n-1]=(1.0)*U[:,n-2]
    V[:,n-1]=(-1.0)*V[:,n-2]
    

    
    return P,U,V

In [ ]:
def Convective_outlet_flow_bc(P,U,V,U_in,P_fix):
    m,n=np.shape(P)
    # Inlet bc
    P[0,:]=P[1,:]
    U[0,:]=2.0*U_in-U[1,:]
    V[0,:]=(-1.0)*V[1,:]
    
    # Outlet bc 
    P[m-1,:]=P_fix
    V[m-1,:]=(1.0)*V[m-2,:]
    
    # Wall 3 bc
    P[:,0]=P[:,1]
    U[:,0]=(1.0)*U[:,1]
    V[:,0]=(-1.0)*V[:,1]
    
    # Wall 4 bc
    P[:,n-1]=P[:,n-2]
    U[:,n-1]=(1.0)*U[:,n-2]
    V[:,n-1]=(-1.0)*V[:,n-2]
    

    
    return P,U,V

def convective_outlet_forcing(U,U_in,dx):
    der_u=np.zeros_like(U)
    m,n=np.shape(U)
    dx_l=0.5*(dx[m-1]+dx[m-2])
    for j in range(1,n-1):
        der_u[m-1,j]=(-1.0*U_in)*(U[m-1,j]-U[m-2,j])/dx_l
    return der_u
    

In [ ]:
def IBM_integrate_time_convec_outlet(P_state,U_state,V_state,U_bound,P_bound,timestep,rho,nu,beta,dx,dy,x_c,y_c,lag_x,lag_y):
    dp,du1,dv1=CFD_EDACM(P_state,U_state,V_state,rho,nu,beta,dx,dy)
    du2,dv2=IBM_forcing_der(U_state,V_state,x_c,y_c,lag_x,lag_y,timestep)
    du3=convective_outlet_forcing(U_state,U_bound,dx)
    
    P_temp=P_state+timestep*dp
    U_temp=U_state+timestep*(du1+du2+du3)
    V_temp=V_state+timestep*(dv1+dv2)
    
    P_temp,U_temp,V_temp=Convective_outlet_flow_bc(P_temp,U_temp,V_temp,U_bound,P_bound)
    
    dp,du1,dv1=CFD_EDACM(P_temp,U_temp,V_temp,rho,nu,beta,dx,dy)
    du2,dv2=IBM_forcing_der(U_temp,V_temp,x_c,y_c,lag_x,lag_y,timestep)
    du3=convective_outlet_forcing(U_temp,U_bound,dx)
    
    P_new=0.5*(P_state+P_temp)+0.5*timestep*dp
    U_new=0.5*(U_state+U_temp)+0.5*timestep*(du1+du2+du3)
    V_new=0.5*(V_state+V_temp)+0.5*timestep*(dv1+dv2)
    
    P_new,U_new,V_new=Convective_outlet_flow_bc(P_new,U_new,V_new,U_bound,P_bound)
    
    return P_new,U_new,V_new

In [ ]:
def create_euler_domains(params):
    dim=params["dim"]
    factor_conc=params["conc_region"]
    factor_inlet=params["in_region"]
    factor_outlet=params["out_region"]
    factor_wall=params["wall_region"]
    div_conc=params["step_conc"] # Number of divisions in conc region
    div_in=params["step_in"] # Number of divisions in inlet region
    div_out=params["step_out"] # Number of divisions in outlet region
    div_wall=params["step_wall"] # Number of divisions in wall region
    
    x=np.concatenate([np.linspace(-1.0*factor_inlet*dim,-1.0*factor_conc*dim,div_in),np.linspace(-1.0*factor_conc*dim,1.0*factor_conc*dim,div_conc)[1:-1],np.linspace(1.0*factor_conc*dim,1.0*factor_outlet*dim,div_out)])
    y=np.concatenate([np.linspace(-1.0*factor_wall*dim,-1.0*factor_conc*dim,div_wall),np.linspace(-1.0*factor_conc*dim,1.0*factor_conc*dim,div_conc)[1:-1],np.linspace(1.0*factor_conc*dim,1.0*factor_wall*dim,div_wall)])
    
    return x,y

In [ ]:
 def create_circle(radius,div):
        dang=div/radius
        ang=np.arange(0,2*np.pi-0.1*dang,dang)
        x=radius*np.cos(ang)
        y=radius*np.sin(ang)
        return x,y

# Now we try flow past a cylinder

In [ ]:
radius=0.75
dim=2.0*radius
Re=100.0
rho=1.0
beta=400.0
U_in=5.0
P_out=0.0
params=dict()
params["dim"]=dim
params["conc_region"]=2.5
params["in_region"]=8.0
params["out_region"]=15.0
params["wall_region"]=6.0
params["step_conc"]=80
params["step_in"]=60
params["step_out"]=200
params["step_wall"]=50
x_eu,y_eu=create_euler_domains(params)

In [ ]:
x_c,y_c,dx,dy,xx,yy,P,U,V=init_domains(x_eu,y_eu,U_in,P_out)
lag_x,lag_y=create_circle(radius,min(dx))

In [ ]:
timestep=1e-5
n_iter=800000
n_print=1000
n_save=80000

In [ ]:
nu=U_in*dim/Re

In [ ]:
P_array=[]
U_array=[]
V_array=[]
for i in range(n_iter):
    P,U,V=IBM_integrate_time_convec_outlet(P,U,V,U_in,P_out,timestep,rho,nu,beta,dx,dy,x_c,y_c,lag_x,lag_y)
    if i%n_print==0:
        print("Iteration "+str(i+1)+"completed")
        div_mat=compute_divergence(U,V,dx,dy)
        print("The maximum value of divergence is :"+str(np.max(np.abs(div_mat))))
        fx,fy=IBM_force_computation(U,V,x_c,y_c,lag_x,lag_y,timestep,rho)
        print("The force in the x direction is: "+str(fx))
        print("The force in the y direction is: "+str(fy))
    if i%n_save==0:
        print("Iteration "+str(i+1)+"saved")
        P_array.append(np.copy(P))
        U_array.append(np.copy(U))
        V_array.append(np.copy(V))


In [ ]:
def plotter(ind):
    plt.figure(figsize=(16,8))
    plt.contourf(xx,yy,U_array[ind],levels=20)
    #plt.streamplot(xx.T,yy.T,U_array[ind].T,V_array[ind].T)
    plt.colorbar()
    
D=interact(plotter,ind=IntSlider(min=0,max=len(U_array)-1,step=1))

# Now we try flow past airfoil

In [ ]:
def NACA_sym_airfoil(chord_length,spacing,thickness):
    t=0.01*thickness*chord_length
    x=np.arange(0,chord_length-0.1*spacing,spacing)
    x_n=x/chord_length
    factor_root=5*0.2969
    factor_1=5*(-0.1260)
    factor_2=5*(-0.3516)
    factor_3=5*(0.2843)
    factor_4=5*(-0.1015)
    y=t*(factor_root*(x_n**0.5)+factor_1*x_n+factor_2*(x_n**2)+factor_3*(x_n**3)+factor_4*(x_n**4))
    lag_x=np.concatenate([x,x])
    lag_y=np.concatenate([y,-y])
    return lag_x,lag_y

In [ ]:
radius=0.75
dim=2.0*radius
Re=100.0
rho=1.0
beta=400.0
U_in=5.0
P_out=0.0
params=dict()
params["dim"]=dim
params["conc_region"]=2.5
params["in_region"]=8.0
params["out_region"]=15.0
params["wall_region"]=6.0
params["step_conc"]=120
params["step_in"]=60
params["step_out"]=200
params["step_wall"]=50
x_eu,y_eu=create_euler_domains(params)

In [ ]:
x_c,y_c,dx,dy,xx,yy,P,U,V=init_domains(x_eu,y_eu,U_in,P_out)
lag_x,lag_y=NACA_sym_airfoil(dim,min(dx),12)

In [ ]:
dy

In [ ]:
timestep=1e-5
n_iter=800000
n_print=1000
n_save=80000
nu=U_in*dim/Re

In [ ]:
P_array=[]
U_array=[]
V_array=[]
for i in range(n_iter):
    P,U,V=IBM_integrate_time_convec_outlet(P,U,V,U_in,P_out,timestep,rho,nu,beta,dx,dy,x_c,y_c,lag_x,lag_y)
    if i%n_print==0:
        print("Iteration "+str(i+1)+"completed")
        div_mat=compute_divergence(U,V,dx,dy)
        print("The maximum value of divergence is :"+str(np.max(np.abs(div_mat))))
        fx,fy=IBM_force_computation(U,V,x_c,y_c,lag_x,lag_y,timestep,rho)
        print("The force in the x direction is: "+str(fx))
        print("The force in the y direction is: "+str(fy))
    if i%n_save==0:
        print("Iteration "+str(i+1)+"saved")
        P_array.append(np.copy(P))
        U_array.append(np.copy(U))
        V_array.append(np.copy(V))


In [ ]:
def plotter(ind):
    plt.figure(figsize=(16,8))
    plt.contourf(xx,yy,U_array[ind],levels=20)
    #plt.streamplot(xx.T,yy.T,U_array[ind].T,V_array[ind].T)
    plt.colorbar()
    
D=interact(plotter,ind=IntSlider(min=0,max=len(U_array)-1,step=1))

In [ ]:
%matplotlib auto

In [ ]:
def CD(Re):
    return (24/Re)*(1+0.15*(Re**0.687))

In [ ]:
CD(100)